# 04 - Demo end-to-end

Flujo completo en AWS:
1. Subir imagen del test set a `s3://mlemons-raw/inference/<lote>/img.jpg`.
2. Disparar Step Function con `{lote_id, images: [...]}`.
3. SFN invoca Lambda `invoke_modelo_a` -> endpoint SageMaker -> predicciones.
4. SFN invoca Lambda `compute_score` -> score + DynamoDB write.
5. Leer registro en DynamoDB y validar.

Pre-requisitos: notebook 03 ejecutado y Model Package aprobado, `terraform apply` con `-var model_package_arn=<arn>`.

In [ ]:
import json
import os
import subprocess
import time
import uuid
import boto3

creds = subprocess.check_output(
    ['aws', 'configure', 'export-credentials', '--format', 'env-no-export']
).decode()
for line in creds.strip().split('\n'):
    k, _, v = line.partition('=')
    os.environ[k] = v

region = 'us-east-2'
sess = boto3.Session(region_name=region)
account = sess.client('sts').get_caller_identity()['Account']

s3 = sess.client('s3')
sfn = sess.client('stepfunctions')
dynamodb = sess.resource('dynamodb')

BUCKET_RAW = f'mlemons-raw-{account}'
STATE_MACHINE_ARN = f'arn:aws:states:{region}:{account}:stateMachine:mlemons-pipeline'
TABLE = dynamodb.Table('mlemons_lotes')
print('Bucket   :', BUCKET_RAW)
print('SFN ARN  :', STATE_MACHINE_ARN)

## Seleccionar imagenes del test set

Tomo 4 buenas y 2 malas del dataset local para simular un lote mixto.

In [ ]:
import glob
import random
random.seed(42)

good = sorted(glob.glob('../data/lemons/good_quality/*.jpg'))
bad  = sorted(glob.glob('../data/lemons/bad_quality/*.jpg'))
muestras = random.sample(good, 4) + random.sample(bad, 2)
print('Imagenes seleccionadas:', len(muestras))
for p in muestras:
    print(' ', p)

## Subir imagenes a S3 y armar evento

In [ ]:
lote_id = f'demo-{uuid.uuid4().hex[:8]}'
images_s3 = []
for path in muestras:
    name = os.path.basename(path)
    key = f'inference/{lote_id}/{name}'
    s3.upload_file(path, BUCKET_RAW, key)
    images_s3.append(f's3://{BUCKET_RAW}/{key}')
print('Lote id:', lote_id)
print('Imagenes:')
for u in images_s3:
    print(' ', u)

## Disparar Step Function

In [ ]:
payload = {'lote_id': lote_id, 'images': images_s3, 'threshold': 75}
resp = sfn.start_execution(
    stateMachineArn=STATE_MACHINE_ARN,
    name=lote_id,
    input=json.dumps(payload),
)
execution_arn = resp['executionArn']
print('Execution ARN:', execution_arn)

## Esperar a que termine

In [ ]:
for _ in range(60):
    desc = sfn.describe_execution(executionArn=execution_arn)
    if desc['status'] != 'RUNNING':
        break
    time.sleep(2)
print('Status:', desc['status'])
print('Output:', desc.get('output'))

## Validar registro en DynamoDB

In [ ]:
item = TABLE.get_item(Key={'lote_id': lote_id}).get('Item')
assert item is not None, 'No se persistio el lote en DynamoDB'
for k, v in item.items():
    print(f'  {k:14s} = {v}')

## Cleanup opcional

Si vas a re-correr el demo, borrar la entrada del lote y las imagenes en S3.

In [ ]:
# TABLE.delete_item(Key={'lote_id': lote_id})
# for u in images_s3:
#     key = u.replace(f's3://{BUCKET_RAW}/', '')
#     s3.delete_object(Bucket=BUCKET_RAW, Key=key)
# print('cleanup ok')